# What Makes a Good Book?

![bookstore](bookstore.jpg)

## 📖 The Background

Identifying popular products is incredibly important for e-commerce companies! Popular products generate more revenue and, therefore, play a key role in stock control.


## 💾 Dataset Summary

You've been asked to support an online bookstore by building a model to predict whether a book will be popular or not. They've supplied you with an extensive dataset (`"books.csv"`) containing information about all books they've sold, including:

* `price`
* `popularity` (target variable)
* `review/summary`
* `review/text`
* `review/helpfulness`
    - formatted as: the number of _helpful_ reviews / the number of reviews
* `authors`
* `categories`


## Objectives

You'll need to build a model that predicts whether a book will be rated as popular or not.

They have high expectations of you, so have set a target of at least 70% accuracy! You are free to use as many features as you like, and will need to engineer new features to achieve this level of performance.

In [12]:
#import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

In [13]:
#import data
orig_df = pd.read_csv("books.csv")

#inspect data
display(orig_df.head(3))
display(orig_df.info())

,title,price,review/helpfulness,review/summary,review/text,description,authors,categories,popularity
0,We Band of Angels: The Untold Story of America...,10.88,2/3,A Great Book about women in WWII,I have alway been a fan of fiction books set i...,"In the fall of 1941, the Philippines was a gar...",'Elizabeth Norman','History',Unpopular
1,Prayer That Brings Revival: Interceding for Go...,9.35,0/0,Very helpful book for church prayer groups and...,Very helpful book to give you a better prayer ...,"In Prayer That Brings Revival, best-selling au...",'Yong-gi Cho','Religion',Unpopular
2,The Mystical Journey from Jesus to Christ,24.95,17/19,Universal Spiritual Awakening Guide With Some ...,The message of this book is to find yourself a...,THE MYSTICAL JOURNEY FROM JESUS TO CHRIST Disc...,'Muata Ashby',"'Body, Mind & Spirit'",Unpopular


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15719 entries, 0 to 15718
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               15719 non-null  object 
 1   price               15719 non-null  float64
 2   review/helpfulness  15719 non-null  object 
 3   review/summary      15718 non-null  object 
 4   review/text         15719 non-null  object 
 5   description         15719 non-null  object 
 6   authors             15719 non-null  object 
 7   categories          15719 non-null  object 
 8   popularity          15719 non-null  object 
dtypes: float64(1), object(8)
memory usage: 1.1+ MB


None

In this dataset, there are 9 variables & about 15,700 data points. Fortunately, there are no missing data points. The variables are fairly self explanatory, but the 'review/helpfulness' column has values that aren't easy to analyze & compare across data points. Generally, they are formatted as: [integer]/[integer]; e.g. 0/0, 17/19, the latter integer is always greater than or equal to the former. It is presumed that these values denote the number of _helpful_ reviews / the number of total reviews. It might be better if this variable was redone to be more interpretable. For example, they could be split, & another column could be made indicating what is the percentage of total reviews that were helpful.

The target variable--'popularity'--has two values--"Popular" & "Unpopular". Moreover, there are several categorical variables that will have to be encoded to numerical values if they are used in the modeling process. Of the variables, the following will likely be used as feature variables: 'price', 'review/helpfulness', 'review/summary', 'review/text', 'description', 'categories'.

Before beginning the modeling process, however, some exploratory analyis should be done to get a better understanding of the data.

## EDA

In [14]:
#explore the data

#orig_df.sample(5)
#orig_df['title'].value_counts()
#orig_df['categories'].value_counts()
orig_df['popularity'].value_counts()

popularity
Unpopular    10490
Popular       5229
Name: count, dtype: int64

Notes from EDA:
- There are 6,985 unique books. The most reviewed book, "Eldest (Inheritance, Book 2)," has 90 reviews.
- There are 6,447 authors, implying that some wrote multiple books that are in this dataset.
- Of the 15,719 reviews, 10,490 of them were labelled as "Unpopular" & 5,229 were labelled as "Popular"; about a 2:1 ratio.
- There are 313 unique book categories. There are also some inconsistencies in this variable; "Fiction", "FICTION".


## Preprocessing the data
Prior to generating the model, the data needs some preprocessing in order for machine learning algorithms to be applied. The main step involves encoding the many categorical variables to numerical values. Additionally, the 'review/helpfulness' variable needs to be distinguished more clearly.

In [15]:
df_cleaning_I = orig_df.copy()

#clean up the 'review/helpfulness' column
df_cleaning_I[['# helpful reviews','# reviews']] = df_cleaning_I['review/helpfulness'].str.split('/', expand=True).astype('int')

#Calculate the % of reviews that were helpful,
    #for rows w/0 reviews, the result is null --> fill missing values with 0
df_cleaning_I['% helpful reviews'] = (df_cleaning_I['# helpful reviews'] / df_cleaning_I['# reviews']).fillna(0)

#Remove original 'review/helpfulness' column
df_cleaning_I.drop(columns=['review/helpfulness'], inplace=True)
display(df_cleaning_I.sample(3))

,title,price,review/summary,review/text,description,authors,categories,popularity,# helpful reviews,# reviews,% helpful reviews
5803,Hitler's Last Offensive (Pen & Sword Military ...,12.99,A Reissued Bulge Classic,HITLER'S LAST OFFENSIVE is a thoroughly detail...,This is the full story of the Battle of the Ar...,'Peter Elstob','History',Unpopular,5,5,1.000000
12198,Angels in the Wilderness,24.95,A courageous and oddly tender tale,Award-winning graphic artist Amy Racina went o...,This book is a first-person account of a disas...,'Amy Racina','Travel',Popular,20,22,0.909091
1017,Frommer's Texas (Frommer's Complete),1.89,A decent travel guifde for Texas,This is one of the few guidebooks one can find...,Frommer's. The best trips start here. Experien...,"'David Baird', 'Eric Peterson', 'Neil Edward S...",'Travel',Unpopular,14,16,0.875000


With the 'categories' variable, it should first be cleaned up in terms of capitaization. This reduced the number of unique categories from 313 to 304. Next, a lot of these categories account for very few books each. For example, only 34 categories account for at least 100 books each, & 243 categories account for less than 10 books each. This variable, given that it is non-numerical, will also have to be encoded. In adding so many additional columns, it could produce a model that has a tendency to overfit. As such, the number of unique categories should be lessened to help produce a more accurate model.  
In this case, categories with at least ten books were included. This reduced the number of categories from 304 to 60 & the number of data points from 15,719 to 15,157.
- After performing the modeling process, more categories needed to be excluded in order to obtain a desirable accuracy score. As such, the threshold was altered to use categories with at least 20 books. This reduced the number of categories from 60 to 56 & the number of data points from 15,157 to 15,102.

In [16]:
#fix capitalization inconsistencies in 'categories' column
df_cleaning_I['categories'] = df_cleaning_I['categories'].str.title()

#exclude categories with < 10 books each
df_cleaning_I = df_cleaning_I.groupby('categories').filter(lambda x: len(x) > 20)
display(df_cleaning_I['categories'].nunique())

#encode the categories to numerical values
categories = pd.get_dummies(df_cleaning_I['categories'], drop_first=True)
#attach the encoded columns to the main data, remove the original 'categories' variable
df_cleaning_II = pd.concat([df_cleaning_I, categories], axis=1)
df_cleaning_II.drop(columns=['categories'], inplace=True)

#display(df_cleaning_II.info())
display(df_cleaning_II.sample(3))

56

,title,price,review/summary,review/text,description,authors,popularity,# helpful reviews,# reviews,% helpful reviews,...,'Science','Self-Help','Social Science','Sports & Recreation','Study Aids','Technology & Engineering','Transportation','Travel','True Crime','Young Adult Fiction'
8929,How to Love a Black Man,11.16,Very Insightful!,Very easy to follow and very insightful...It's...,As he sheds light on the hidden emotional psyc...,'Ronn Elmore',Popular,0,0,0.000000,...,False,False,True,False,False,False,False,False,False,False
2224,"A Dog Year: Twelve Months, Four Dogs, and Me",9.45,Katz Kills Dogs,"Before you buy this book, you need to know tha...","“Change loves me, defines and stalks me like a...",'Jon Katz',Unpopular,46,56,0.821429,...,False,False,False,False,False,False,False,False,False,False
2077,Anyone But You,39.25,Satisfying Fluff,"If I keep reading Jennifer Crusie, I'm going t...","Things are going great for Sutton Kay, or at l...",'Chelsea M. Cameron',Unpopular,0,0,0.000000,...,False,False,False,False,False,False,False,False,False,False


Now, the next step involves encoding the non-numerical variables that will be used as feature variables in the modeling process. With variables that contain excerpts of text of varying lengths, such as 'review/summary' & 'review/text', one way to utilize them in predictive modeling is to create a matrix similar to one-hot encoding for categorical variables. Columns will designate words found within all of the text values from the original variable, then the values will indicate how many times those words are present in that data point. For example, in a data point in th 'description' column, if the word "first" is present five times, then the cell in this hypothetical matrix will equate to 5.

The `CountVectorizer()` function from sklearn is capable of this. It compiles the words (or tokens) in variables with text & determines how many times each word is present in a data point. Moreover, this function can take specific inputted words to look for & include in the resulting matrix. This is relevant here because certain words may be more indicative of a popular or unpopular book. For example, in a review of a popular book, someone may utilize words like "perfect," "wonderful," "excellent," etc.

To make sure that words are not overcounted, it helps to make them as consistent as possible. More specifically, the words should have consistent capitalization. As to what words should be included here, they can be compiled in a variety of ways. In this case, about a couple dozen words were compiled & used that are indicative of positive reviews that might be associated with a "popular" book. These words are available in the `pos_words` variable.

In [17]:
#ensure capitalization of words across text variables is consistent
for col in ['review/summary','review/text','description']:
    df_cleaning_II[col] = df_cleaning_II[col].str.lower()

#compile words likely associated with reviews of "popular" books
pos_words = ["amazing", "beautiful", "brilliant", "enjoy", "excellent", "exceptional", "fantastic", "good",
             "great", "helpful", "impressive", "incredible", "interesting", "like", "love", "outstanding",
             "perfect", "positive", "remarkable", "thrilling", "useful", "wonderful"]

In [18]:
#instantiate a CountVectorizer
vectorizer = CountVectorizer(vocabulary = pos_words)

#fit, transform 'review/summary', 'review/text', 'description'
rev_sum_transformed = vectorizer.fit_transform(df_cleaning_II['review/summary'].fillna(''))
rev_text_transformed = vectorizer.fit_transform(df_cleaning_II['review/text'])
desc_transformed = vectorizer.fit_transform(df_cleaning_II['description'])

#obtain counts of pos_words from 'review/summary', 'review/text', 'description'
df_cleaning_III = df_cleaning_II.copy()
df_cleaning_III['pos_words_summary'] = rev_sum_transformed.sum(axis=1).reshape(-1,1)
df_cleaning_III['pos_words_text'] = rev_text_transformed.sum(axis=1).reshape(-1,1)
df_cleaning_III['pos_words_desc'] = desc_transformed.sum(axis=1).reshape(-1,1)

#remove original columns
df_cleaning_III.drop(columns=['review/summary', 'review/text', 'description'], inplace=True)

display(df_cleaning_III.info())

<class 'pandas.core.frame.DataFrame'>
Index: 15102 entries, 0 to 15718
Data columns (total 65 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   title                          15102 non-null  object 
 1   price                          15102 non-null  float64
 2   authors                        15102 non-null  object 
 3   popularity                     15102 non-null  object 
 4   # helpful reviews              15102 non-null  int32  
 5   # reviews                      15102 non-null  int32  
 6   % helpful reviews              15102 non-null  float64
 7   'Architecture'                 15102 non-null  bool   
 8   'Art'                          15102 non-null  bool   
 9   'Bibles'                       15102 non-null  bool   
 10  'Biography & Autobiography'    15102 non-null  bool   
 11  'Body, Mind & Spirit'          15102 non-null  bool   
 12  'Books And Reading'            15102 non-null  bool

None

## Modeling
Now that the data has been preprocessed appropriately, it can be used for modeling (don't forget to remove the 'title', 'authors' variables).

Given the complexity of this task & the dataset, a random forest classification algorithm was utilized. A random-search cross validation was also utilized to tune some hyperparameters.
- Since the the dataset is imbalanced, particularly in the 'categories' variable, it will help to apply weights to counter the imbalance(s). To do so, can set 'class_weight' to "balanced".

In [19]:
#split data into feature / target data
X = df_cleaning_III.drop(columns=['title','authors','popularity']).values
y = df_cleaning_III['popularity'].values.reshape(-1,1)

#split data into training/testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=20)

**NOTE** that the cell below can take several minutes to execute.

In [21]:
#generate hyperparameter values to trial
params = {'n_estimators':np.arange(100,201,20), 'max_depth':np.arange(10,101,10), 'min_samples_split':np.arange(2,6,1)}

#instantiate the classifier
rf_raw = RandomForestClassifier(class_weight="balanced", random_state=20)

#perform a 5-fold random-search cross validation
rand_CV = RandomizedSearchCV(rf_raw, param_distributions=params, cv=5, random_state=20)

#fit the CV to the data
    #.ravel() is used to flatten arrays
rand_CV.fit(X_train, y_train.ravel())

RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    random_state=20),
                   param_distributions={'max_depth': array([ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100]),
                                        'min_samples_split': array([2, 3, 4, 5]),
                                        'n_estimators': array([100, 120, 140, 160, 180, 200])},
                   random_state=20)

In [22]:
#obtain the best hyperparameter values
hyp_params = rand_CV.best_params_
display(hyp_params)

{'n_estimators': 100, 'min_samples_split': 2, 'max_depth': 60}

With a five-fold random-search cross validation, the optimal hyperparameters values are:
- `n_estimators` = 100
- `max_depth` = 60
- `min_samples_split` = 2

These values will be used to create the final random forest model.

In [23]:
#Create the random forest model using the optimal hyperparameter values
rf = RandomForestClassifier(n_estimators=hyp_params['n_estimators'], max_depth=hyp_params['max_depth'], 
                            min_samples_split=hyp_params['min_samples_split'], class_weight="balanced", random_state=42)

#fit the data to the model
rf.fit(X_train, y_train.ravel())

RandomForestClassifier(class_weight='balanced', max_depth=60, random_state=42)

In [24]:
#Evaluate model accuracy
display(rf.score(X_train, y_train))
display(rf.score(X_test, y_test))

model_accuracy = rf.score(X_test, y_test)

0.9862263817764436

0.703125

The model had an accuracy score of about 70.3% on the test data, which exceeds the 70% goal.